# PropertyLens Feature Engineering (11-factor coverage)

This notebook migrates feature-building logic into `02_feature_layer` and produces a training-ready feature table for HDB resale price prediction.

Covered factors:
1. level
2. lease remaining years
3. size
4. room count
5. distance to MRT
6. orientation / facing road score
7. distance to highway
8. distance to foodcourt
9. large commercial access
10. all-school proximity (distance + density)
11. primary-school tier score within 1km (implicit quality from enrollment competition)

In [13]:
import hashlib
import json
import math
import os
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

SEED = 42
np.random.seed(SEED)

ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
RAW_DIR = ROOT / '01_data_layer' / 'raw'
GEO_DIR = RAW_DIR / 'google_geo'
SCHOOL_DIR = RAW_DIR / 'schools'
FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
OUTPUT_DIR = FEATURE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime('%Y%m%d')

print('ROOT:', ROOT)
print('RAW_DIR:', RAW_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

ROOT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens
RAW_DIR: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw
OUTPUT_DIR: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training/outputs


In [14]:
def load_hdb_2015_plus() -> pd.DataFrame:
    hdb_dir = RAW_DIR / 'ResaleFlatPrices'
    files = sorted(hdb_dir.glob('*.csv'))
    if not files:
        raise FileNotFoundError(f'No HDB CSV found in {hdb_dir}')

    parts = []
    for fp in files:
        d = pd.read_csv(fp)
        d['month_dt'] = pd.to_datetime(d['month'], errors='coerce')
        d = d[d['month_dt'].dt.year >= 2015].copy()
        d['source_file'] = fp.name
        parts.append(d)

    hdb = pd.concat(parts, ignore_index=True)
    hdb['address_key'] = (
        hdb['block'].astype(str).str.strip() + ' ' + hdb['street_name'].astype(str).str.strip()
    ).str.upper()
    hdb['transaction_year'] = hdb['month_dt'].dt.year

    # Use the same source_id convention as raw collection notebook for stable joining
    hdb['source_id'] = hdb.apply(
        lambda r: hashlib.md5(
            f"{str(r['block']).strip()}|{str(r['street_name']).strip()}|{str(r['town']).strip()}".encode('utf-8')
        ).hexdigest(),
        axis=1,
    )

    hdb['resale_price'] = pd.to_numeric(hdb['resale_price'], errors='coerce')
    hdb['floor_area_sqm'] = pd.to_numeric(hdb['floor_area_sqm'], errors='coerce')
    hdb['lease_commence_date'] = pd.to_numeric(hdb['lease_commence_date'], errors='coerce')

    # Factor 1: level (midpoint of storey range)
    level_bounds = hdb['storey_range'].astype(str).str.extract(r'(\d+)\s+TO\s+(\d+)')
    hdb['level_mid'] = (
        pd.to_numeric(level_bounds[0], errors='coerce') + pd.to_numeric(level_bounds[1], errors='coerce')
    ) / 2

    # Factor 2: lease remaining years, assuming 99-year lease
    hdb['lease_remaining_years'] = 99 - (hdb['transaction_year'] - hdb['lease_commence_date'])
    hdb['lease_remaining_years'] = hdb['lease_remaining_years'].clip(lower=0, upper=99)

    # Factor 4: room count from flat_type
    room_num = hdb['flat_type'].astype(str).str.extract(r'(\d+)')
    hdb['room_count'] = pd.to_numeric(room_num[0], errors='coerce')
    hdb.loc[hdb['flat_type'].astype(str).str.contains('EXECUTIVE', case=False, na=False), 'room_count'] = 5
    hdb.loc[hdb['flat_type'].astype(str).str.contains('MULTI', case=False, na=False), 'room_count'] = 6

    return hdb


def normalize_address_from_requested(v: pd.Series) -> pd.Series:
    return (
        v.astype(str)
        .str.replace(r',\s*SINGAPORE\s*$', '', regex=True)
        .str.strip()
        .str.upper()
    )


def latest_file_by_pattern(folder: Path, pattern: str) -> Path | None:
    files = sorted(folder.glob(pattern))
    return files[-1] if files else None


hdb = load_hdb_2015_plus()
print('HDB rows (2015+):', len(hdb))
hdb[['month', 'town', 'flat_type', 'storey_range', 'floor_area_sqm', 'resale_price', 'level_mid', 'lease_remaining_years', 'room_count']].head()

HDB rows (2015+): 526848


,month,town,flat_type,storey_range,floor_area_sqm,resale_price,level_mid,lease_remaining_years,room_count
0,2015-01,ANG MO KIO,3 ROOM,07 TO 09,60.0,255000.0,8.0,70.0,3.0
1,2015-01,ANG MO KIO,3 ROOM,01 TO 03,68.0,275000.0,2.0,65.0,3.0
2,2015-01,ANG MO KIO,3 ROOM,01 TO 03,69.0,285000.0,2.0,64.0,3.0
3,2015-01,ANG MO KIO,3 ROOM,01 TO 03,68.0,290000.0,2.0,63.0,3.0
4,2015-01,ANG MO KIO,3 ROOM,07 TO 09,68.0,290000.0,8.0,64.0,3.0


In [15]:
# Load geocode/accessibility features prepared in data layer
geo_acc_fp = latest_file_by_pattern(GEO_DIR, 'hdb_geo_accessibility_noise_features_*.csv')
geo_hw_fp = latest_file_by_pattern(GEO_DIR, 'onemap_hdb_geocode_with_highway_dist_*.csv')

if geo_acc_fp is None:
    raise FileNotFoundError('Missing hdb_geo_accessibility_noise_features_*.csv in raw/google_geo')
if geo_hw_fp is None:
    raise FileNotFoundError('Missing onemap_hdb_geocode_with_highway_dist_*.csv in raw/google_geo')

geo_acc = pd.read_csv(geo_acc_fp)
geo_hw = pd.read_csv(geo_hw_fp)

# Primary join path: source_id (stable key from raw collection)
acc_keep = [
    'source_id',
    'lat', 'lng',
    'nearest_mrt_km',
    'road_noise_score',
    'facing_road_noise_proxy',
]
acc_keep = [c for c in acc_keep if c in geo_acc.columns]
geo_acc_slim = geo_acc[acc_keep].drop_duplicates('source_id')

hw_keep = ['source_id', 'highway_distance_km']
hw_keep = [c for c in hw_keep if c in geo_hw.columns]
geo_hw_slim = geo_hw[hw_keep].drop_duplicates('source_id')

feat = hdb.merge(geo_acc_slim, on='source_id', how='left').merge(geo_hw_slim, on='source_id', how='left')

# Fallback join for any unmatched rows via normalized address key
missing_mask = feat['lat'].isna() | feat['lng'].isna()
if missing_mask.any():
    geo_acc_addr = geo_acc.copy()
    geo_hw_addr = geo_hw.copy()
    geo_acc_addr['address_key'] = normalize_address_from_requested(geo_acc_addr['requested_address'])
    geo_hw_addr['address_key'] = normalize_address_from_requested(geo_hw_addr['requested_address'])

    geo_acc_addr = geo_acc_addr[['address_key', 'lat', 'lng', 'nearest_mrt_km', 'road_noise_score', 'facing_road_noise_proxy']].drop_duplicates('address_key')
    geo_hw_addr = geo_hw_addr[['address_key', 'highway_distance_km']].drop_duplicates('address_key')

    fallback = feat.loc[missing_mask, ['address_key']].merge(geo_acc_addr, on='address_key', how='left').merge(geo_hw_addr, on='address_key', how='left')
    for c in ['lat', 'lng', 'nearest_mrt_km', 'road_noise_score', 'facing_road_noise_proxy', 'highway_distance_km']:
        if c in fallback.columns:
            feat.loc[missing_mask, c] = feat.loc[missing_mask, c].fillna(fallback[c].values)

# Factor 5: MRT distance (meters)
feat['dist_to_mrt_m'] = pd.to_numeric(feat.get('nearest_mrt_km'), errors='coerce') * 1000

# Factor 6: orientation score proxy (facing road: penalty)
frp = pd.to_numeric(feat.get('facing_road_noise_proxy'), errors='coerce')
rns = pd.to_numeric(feat.get('road_noise_score'), errors='coerce')
feat['facing_road_flag'] = np.where(frp.notna(), (frp > 0.5).astype(int), np.nan)
feat.loc[feat['facing_road_flag'].isna(), 'facing_road_flag'] = np.where(rns.notna(), (rns > rns.median()).astype(int), np.nan)
feat['orientation_score'] = np.where(feat['facing_road_flag'] == 1, -1.0, 1.0)

# Factor 7: distance to highway (meters)
feat['dist_to_highway_m'] = pd.to_numeric(feat.get('highway_distance_km'), errors='coerce') * 1000

print('Merged feature base rows:', len(feat))
print('Matched coordinates:', int(feat['lat'].notna().sum()), '/', len(feat))
feat[['address_key', 'lat', 'lng', 'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m']].head()

Merged feature base rows: 526848
Matched coordinates: 526848 / 526848


,address_key,lat,lng,dist_to_mrt_m,orientation_score,dist_to_highway_m
0,174 ANG MO KIO AVE 4,1.375097,103.837619,1176.154764,1.0,NaN
1,541 ANG MO KIO AVE 10,1.373922,103.855621,2557.672559,1.0,NaN
2,163 ANG MO KIO AVE 4,1.373549,103.838176,1356.874106,1.0,NaN
3,446 ANG MO KIO AVE 10,1.367761,103.855357,2904.319336,1.0,NaN
4,557 ANG MO KIO AVE 10,1.371626,103.857736,2891.175626,1.0,NaN


In [16]:
# Collect or load foodcourt and mall POI data when missing in raw layer
DATASTORE_ENDPOINT = 'https://data.gov.sg/api/action/datastore_search'
ONEMAP_SEARCH_ENDPOINT = 'https://www.onemap.gov.sg/api/common/elastic/search'
NEA_HAWKER_RESOURCE_ID = 'd_4a086da0a5553be1d89383cd90d07ecd'

session = requests.Session()

def fetch_datagov_resource(resource_id: str, limit: int = 5000) -> pd.DataFrame:
    offset = 0
    rows = []
    while True:
        try:
            resp = session.get(DATASTORE_ENDPOINT, params={'resource_id': resource_id, 'limit': limit, 'offset': offset}, timeout=30)
            payload = resp.json()
        except Exception:
            break
        if not payload.get('success'):
            break
        batch = payload['result'].get('records', [])
        rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return pd.DataFrame(rows)

def onemap_search_all(search_val: str, max_pages: int = 8) -> pd.DataFrame:
    rows = []
    for page in range(1, max_pages + 1):
        try:
            r = session.get(ONEMAP_SEARCH_ENDPOINT, params={
                'searchVal': search_val,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': page,
            }, timeout=30)
            p = r.json()
        except Exception:
            break
        batch = p.get('results', [])
        if not batch:
            break
        rows.extend(batch)
    return pd.DataFrame(rows)

hawker_fp = latest_file_by_pattern(GEO_DIR, 'nea_hawker_centres_*.csv')
if hawker_fp is None:
    hawker = fetch_datagov_resource(NEA_HAWKER_RESOURCE_ID)
    if len(hawker):
        save_fp = GEO_DIR / f'nea_hawker_centres_{RUN_DATE}.csv'
        hawker.to_csv(save_fp, index=False)
        hawker_fp = save_fp
    else:
        # Fallback via OneMap keyword search
        hawker = onemap_search_all('HAWKER CENTRE', max_pages=12)
        if len(hawker):
            hawker = hawker.rename(columns={'SEARCHVAL': 'name', 'LATITUDE': 'lat', 'LONGITUDE': 'lng', 'ADDRESS': 'address'})
            save_fp = GEO_DIR / f'nea_hawker_centres_{RUN_DATE}.csv'
            hawker.to_csv(save_fp, index=False)
            hawker_fp = save_fp

mall_fp = latest_file_by_pattern(GEO_DIR, 'onemap_mall_nodes_*.csv')
if mall_fp is None:
    mall = onemap_search_all('SHOPPING MALL', max_pages=10)
    if len(mall):
        mall = mall.rename(columns={'SEARCHVAL': 'name', 'LATITUDE': 'lat', 'LONGITUDE': 'lng', 'ADDRESS': 'address'})
        save_fp = GEO_DIR / f'onemap_mall_nodes_{RUN_DATE}.csv'
        mall.to_csv(save_fp, index=False)
        mall_fp = save_fp

print('hawker file:', hawker_fp)
print('mall file:', mall_fp)

hawker file: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/google_geo/nea_hawker_centres_20260317.csv
mall file: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/google_geo/onemap_mall_nodes_20260317.csv


In [17]:
def _to_numeric_lat_lng(df: pd.DataFrame, lat_candidates: list[str], lng_candidates: list[str]) -> pd.DataFrame:
    out = df.copy()
    lat_col = next((c for c in lat_candidates if c in out.columns), None)
    lng_col = next((c for c in lng_candidates if c in out.columns), None)
    if lat_col is None or lng_col is None:
        return pd.DataFrame(columns=['lat', 'lng'])
    out['lat'] = pd.to_numeric(out[lat_col], errors='coerce')
    out['lng'] = pd.to_numeric(out[lng_col], errors='coerce')
    return out.dropna(subset=['lat', 'lng'])

def build_balltree(points_lat_lng: np.ndarray) -> BallTree:
    rad = np.radians(points_lat_lng.astype(float))
    return BallTree(rad, metric='haversine')

def nearest_distance_and_count(home_lat_lng: np.ndarray, poi_lat_lng: np.ndarray, radius_km: float) -> tuple[np.ndarray, np.ndarray]:
    if len(home_lat_lng) == 0 or len(poi_lat_lng) == 0:
        n = len(home_lat_lng)
        return np.full(n, np.nan), np.zeros(n, dtype=int)

    home_rad = np.radians(home_lat_lng.astype(float))
    poi_rad = np.radians(poi_lat_lng.astype(float))
    tree = BallTree(poi_rad, metric='haversine')

    dist_rad, _ = tree.query(home_rad, k=1)
    nearest_km = dist_rad[:, 0] * 6371.0

    idx = tree.query_radius(home_rad, r=radius_km / 6371.0)
    count = np.array([len(i) for i in idx], dtype=int)
    return nearest_km, count


home = feat[['lat', 'lng']].copy()
home['lat'] = pd.to_numeric(home['lat'], errors='coerce')
home['lng'] = pd.to_numeric(home['lng'], errors='coerce')
home_valid_mask = home['lat'].notna() & home['lng'].notna()
home_coords = home.loc[home_valid_mask, ['lat', 'lng']].to_numpy()

# Factor 8: foodcourt distance
if hawker_fp is not None and Path(hawker_fp).exists():
    hawker_df = pd.read_csv(hawker_fp)
else:
    hawker_df = pd.DataFrame()
hawker_geo = _to_numeric_lat_lng(hawker_df, ['lat', 'latitude', 'LATITUDE'], ['lng', 'longitude', 'LONGITUDE', 'longtitude'])
hawker_coords = hawker_geo[['lat', 'lng']].drop_duplicates().to_numpy() if len(hawker_geo) else np.empty((0, 2))

food_nearest_km, _ = nearest_distance_and_count(home_coords, hawker_coords, radius_km=1.0)
feat['dist_to_foodcourt_m'] = np.nan
feat.loc[home_valid_mask, 'dist_to_foodcourt_m'] = food_nearest_km * 1000

# Factor 9: nearby commercial (mall) quality/quantity
if mall_fp is not None and Path(mall_fp).exists():
    mall_df = pd.read_csv(mall_fp)
else:
    mall_df = pd.DataFrame()
mall_geo = _to_numeric_lat_lng(mall_df, ['lat', 'LATITUDE'], ['lng', 'LONGITUDE'])
mall_geo['name'] = mall_df.get('name', mall_df.get('SEARCHVAL', pd.Series(index=mall_geo.index, dtype='object'))).astype(str) if len(mall_geo) else ''
mall_geo = mall_geo.drop_duplicates(subset=['lat', 'lng'])
mall_coords = mall_geo[['lat', 'lng']].to_numpy() if len(mall_geo) else np.empty((0, 2))

mall_nearest_km, mall_count_3km = nearest_distance_and_count(home_coords, mall_coords, radius_km=3.0)
feat['dist_to_nearest_mall_m'] = np.nan
feat['mall_count_3km'] = 0
feat.loc[home_valid_mask, 'dist_to_nearest_mall_m'] = mall_nearest_km * 1000
feat.loc[home_valid_mask, 'mall_count_3km'] = mall_count_3km

# Size proxy: brand keyword upweight, then aggregate weighted accessibility within 3km
if len(mall_geo) and len(home_coords):
    names = mall_geo.get('name', pd.Series('', index=mall_geo.index)).astype(str).str.upper()
    big_kw = names.str.contains('MEGA|HUB|CITY|JUNCTION|POINT|PLAZA|CENTRE', regex=True, na=False)
    mall_weight = np.where(big_kw, 1.5, 1.0)

    home_rad = np.radians(home_coords)
    mall_rad = np.radians(mall_coords.astype(float))
    tree = BallTree(mall_rad, metric='haversine')
    neighbors = tree.query_radius(home_rad, r=3.0 / 6371.0, return_distance=True, sort_results=True)

    weighted_access = []
    for idx_arr, dist_arr in zip(neighbors[0], neighbors[1]):
        if len(idx_arr) == 0:
            weighted_access.append(0.0)
            continue
        d_km = np.maximum(dist_arr * 6371.0, 0.05)
        score = np.sum(mall_weight[idx_arr] / (d_km + 0.25))
        weighted_access.append(float(score))

    feat['mall_weighted_access_3km'] = 0.0
    feat.loc[home_valid_mask, 'mall_weighted_access_3km'] = weighted_access
else:
    feat['mall_weighted_access_3km'] = 0.0

print('Hawker points:', len(hawker_coords), '| Mall points:', len(mall_coords))

Hawker points: 61 | Mall points: 158


In [18]:
# Factor 10 & 11: school proximity + primary school implicit quality tier
moe_fp = latest_file_by_pattern(SCHOOL_DIR, 'moe_general_information_of_schools_*.csv')
sg_fp = latest_file_by_pattern(SCHOOL_DIR, 'sgschooling_2015plus_*.csv')

if moe_fp is None or sg_fp is None:
    raise FileNotFoundError('Missing MOE or sgschooling school datasets in raw/schools')

moe = pd.read_csv(moe_fp)
sg = pd.read_csv(sg_fp)

def build_primary_quality(df_sg: pd.DataFrame) -> pd.DataFrame:
    s = df_sg.copy()
    s['school'] = s['school'].astype(str).str.strip()
    s = s[s['school'].str.len() > 0]

    # Keep school-level rows and clean numeric signals
    s = s[~s['school'].str.startswith('↳', na=False)].copy()
    s['school_upper'] = s['school'].str.upper()

    for col in ['competition_ratio_extracted', 'phase_1', 'applicants_extracted', 'vacancies_extracted']:
        if col in s.columns:
            s[col] = pd.to_numeric(s[col], errors='coerce')

    comp = s['competition_ratio_extracted'] if 'competition_ratio_extracted' in s.columns else pd.Series(np.nan, index=s.index)
    phase1 = s['phase_1'] if 'phase_1' in s.columns else pd.Series(np.nan, index=s.index)
    applicants = s['applicants_extracted'] if 'applicants_extracted' in s.columns else pd.Series(np.nan, index=s.index)
    vacancies = s['vacancies_extracted'] if 'vacancies_extracted' in s.columns else pd.Series(np.nan, index=s.index)

    implied_ratio = applicants / vacancies.replace(0, np.nan)
    raw_quality = np.nanmax(np.vstack([comp.fillna(np.nan), implied_ratio.fillna(np.nan)]), axis=0)
    raw_quality = pd.Series(raw_quality, index=s.index).fillna(1.0)

    # Early phase demand as extra quality signal
    phase1_norm = phase1.fillna(phase1.median() if phase1.notna().any() else 0.0)
    quality = 0.7 * raw_quality + 0.3 * (phase1_norm / (phase1_norm.max() if phase1_norm.max() else 1.0))

    out = pd.DataFrame({'school_upper': s['school_upper'], 'raw_quality': quality})
    out = out.groupby('school_upper', as_index=False)['raw_quality'].mean()

    qmin = out['raw_quality'].min()
    qmax = out['raw_quality'].max()
    out['school_quality_score'] = 100 * (out['raw_quality'] - qmin) / (qmax - qmin + 1e-9)
    out['school_tier'] = pd.cut(out['school_quality_score'], bins=[-1, 40, 70, 100], labels=['C', 'B', 'A'])
    return out

primary_quality = build_primary_quality(sg)

# Primary school set from MOE
moe['school_name_upper'] = moe['school_name'].astype(str).str.strip().str.upper()
primary = moe[moe['mainlevel_code'].astype(str).str.upper() == 'PRIMARY'].copy()
primary = primary.merge(primary_quality, left_on='school_name_upper', right_on='school_upper', how='left')

# Geocode all schools (cache in raw/google_geo)
school_geo_fp = latest_file_by_pattern(GEO_DIR, 'moe_school_geocode_*.csv')
if school_geo_fp is not None:
    school_geo_raw = pd.read_csv(school_geo_fp)
else:
    rows = []
    endpoint = 'https://www.onemap.gov.sg/api/common/elastic/search'
    for _, r in moe[['school_name', 'address']].drop_duplicates().iterrows():
        query = str(r['address']) if pd.notna(r['address']) and str(r['address']).strip() else str(r['school_name'])
        try:
            resp = session.get(endpoint, params={
                'searchVal': query,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': 1,
            }, timeout=25).json()
            result = resp.get('results', [])
            if result:
                rows.append({
                    'school_name': r['school_name'],
                    'address': r['address'],
                    'lat': result[0].get('LATITUDE'),
                    'lng': result[0].get('LONGITUDE'),
                })
        except Exception:
            continue

    school_geo_raw = pd.DataFrame(rows)
    if len(school_geo_raw):
        school_geo_fp = GEO_DIR / f'moe_school_geocode_{RUN_DATE}.csv'
        school_geo_raw.to_csv(school_geo_fp, index=False)

if len(school_geo_raw) == 0:
    school_geo = pd.DataFrame(columns=['school_name', 'lat', 'lng'])
else:
    school_geo = school_geo_raw.copy()
    school_geo['lat'] = pd.to_numeric(school_geo['lat'], errors='coerce')
    school_geo['lng'] = pd.to_numeric(school_geo['lng'], errors='coerce')
    school_geo = school_geo.dropna(subset=['lat', 'lng']).drop_duplicates(subset=['school_name'])

# All-school proximity
all_school_coords = school_geo[['lat', 'lng']].drop_duplicates().to_numpy() if len(school_geo) else np.empty((0, 2))
all_school_nearest_km, all_school_count_1km = nearest_distance_and_count(home_coords, all_school_coords, radius_km=1.0)
feat['dist_to_nearest_school_m'] = np.nan
feat['school_count_1km'] = 0
feat.loc[home_valid_mask, 'dist_to_nearest_school_m'] = all_school_nearest_km * 1000
feat.loc[home_valid_mask, 'school_count_1km'] = all_school_count_1km

# Primary-tier features within 1km
primary_geo = primary[['school_name', 'school_quality_score']].merge(
    school_geo[['school_name', 'lat', 'lng']],
    on='school_name',
    how='left',
)
primary_geo['lat'] = pd.to_numeric(primary_geo['lat'], errors='coerce')
primary_geo['lng'] = pd.to_numeric(primary_geo['lng'], errors='coerce')
primary_geo = primary_geo.dropna(subset=['lat', 'lng'])

if len(primary_geo) and len(home_coords):
    pq = primary_geo['school_quality_score'].fillna(primary_geo['school_quality_score'].median() if primary_geo['school_quality_score'].notna().any() else 50.0).to_numpy()
    pcoords = primary_geo[['lat', 'lng']].to_numpy()

    home_rad = np.radians(home_coords)
    p_rad = np.radians(pcoords)
    tree = BallTree(p_rad, metric='haversine')
    idx, dist = tree.query_radius(home_rad, r=1.0 / 6371.0, return_distance=True, sort_results=True)

    q_mean = []
    q_top = []
    q_cnt = []
    for ids, d in zip(idx, dist):
        if len(ids) == 0:
            q_mean.append(np.nan)
            q_top.append(np.nan)
            q_cnt.append(0)
            continue
        w = 1.0 / np.maximum(d * 6371.0, 0.05)
        q = pq[ids]
        q_mean.append(float(np.average(q, weights=w)))
        q_top.append(float(np.max(q)))
        q_cnt.append(int(len(ids)))

    feat['primary_school_quality_1km_weighted'] = np.nan
    feat['primary_school_top_quality_1km'] = np.nan
    feat['primary_school_count_1km'] = 0
    feat.loc[home_valid_mask, 'primary_school_quality_1km_weighted'] = q_mean
    feat.loc[home_valid_mask, 'primary_school_top_quality_1km'] = q_top
    feat.loc[home_valid_mask, 'primary_school_count_1km'] = q_cnt
else:
    feat['primary_school_quality_1km_weighted'] = np.nan
    feat['primary_school_top_quality_1km'] = np.nan
    feat['primary_school_count_1km'] = 0

print('Schools geocoded file:', school_geo_fp)
print('All-school points:', len(all_school_coords))
print('Primary schools with coords:', len(primary_geo))

/var/folders/6p/nwg1ljcj77bfnrwnqw_yprf80000gn/T/ipykernel_55416/818766814.py:30: RuntimeWarning: All-NaN slice encountered
  raw_quality = np.nanmax(np.vstack([comp.fillna(np.nan), implied_ratio.fillna(np.nan)]), axis=0)


Schools geocoded file: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/google_geo/moe_school_geocode_20260317.csv
All-school points: 334
Primary schools with coords: 179


In [19]:
# Final cleanup, categorical encoding, and temporal split export
base_cols = [
    'resale_price',
    'transaction_year',
    'town',
    'flat_type',
    'flat_model',
    'level_mid',
    'lease_remaining_years',
    'floor_area_sqm',
    'room_count',
    'dist_to_mrt_m',
    'orientation_score',
    'dist_to_highway_m',
    'dist_to_foodcourt_m',
    'dist_to_nearest_mall_m',
    'mall_count_3km',
    'mall_weighted_access_3km',
    'dist_to_nearest_school_m',
    'school_count_1km',
    'primary_school_quality_1km_weighted',
    'primary_school_top_quality_1km',
    'primary_school_count_1km',
]

use = feat[base_cols + ['address_key']].copy()

for c in [
    'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count',
    'dist_to_mrt_m', 'dist_to_highway_m', 'dist_to_foodcourt_m',
    'dist_to_nearest_mall_m', 'mall_count_3km', 'mall_weighted_access_3km',
    'dist_to_nearest_school_m', 'school_count_1km',
    'primary_school_quality_1km_weighted', 'primary_school_top_quality_1km', 'primary_school_count_1km',
]:
    use[c] = pd.to_numeric(use[c], errors='coerce')

# Robust fill
for c in use.columns:
    if c in ['town', 'flat_type', 'flat_model', 'address_key']:
        use[c] = use[c].astype(str).fillna('UNKNOWN')
    elif c != 'resale_price':
        med = use[c].median() if use[c].notna().any() else 0
        use[c] = use[c].fillna(med)

use = use.dropna(subset=['resale_price'])

cat_cols = ['town', 'flat_type', 'flat_model']
use_model = pd.get_dummies(use, columns=cat_cols, drop_first=True)

# Keep a deterministic temporal split (20% test; half from latest year, half from history)
latest_year = int(use_model['transaction_year'].max())
idx_latest = use_model[use_model['transaction_year'] == latest_year].index.to_numpy()
idx_hist = use_model[use_model['transaction_year'] < latest_year].index.to_numpy()

test_n = max(1, int(len(use_model) * 0.2))
test_latest_n = min(len(idx_latest), max(1, test_n // 2))
test_hist_n = min(len(idx_hist), test_n - test_latest_n)

rng = np.random.default_rng(SEED)
test_idx = np.concatenate([
    rng.choice(idx_latest, size=test_latest_n, replace=False) if test_latest_n > 0 else np.array([], dtype=int),
    rng.choice(idx_hist, size=test_hist_n, replace=False) if test_hist_n > 0 else np.array([], dtype=int),
])
test_idx = np.unique(test_idx)

# Optimized: use vectorized isin instead of slow list comprehension
train_mask = ~np.isin(use_model.index.values, test_idx)
train_idx = use_model.index[train_mask]

train_df = use_model.loc[train_idx].reset_index(drop=True)
test_df = use_model.loc[test_idx].reset_index(drop=True)

feature_table_fp = OUTPUT_DIR / f'hdb_feature_table_{RUN_DATE}.csv'
train_fp = OUTPUT_DIR / f'hdb_feature_train_{RUN_DATE}.csv'
test_fp = OUTPUT_DIR / f'hdb_feature_test_{RUN_DATE}.csv'

use_model.to_csv(feature_table_fp, index=False)
train_df.to_csv(train_fp, index=False)
test_df.to_csv(test_fp, index=False)

meta = {
    'run_date': RUN_DATE,
    'rows_total': int(len(use_model)),
    'rows_train': int(len(train_df)),
    'rows_test': int(len(test_df)),
    'latest_year': latest_year,
    'feature_count': int(use_model.shape[1] - 1),
    'target_col': 'resale_price',
    'core_factor_coverage': [
        'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count',
        'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m', 'dist_to_foodcourt_m',
        'mall_count_3km', 'mall_weighted_access_3km',
        'dist_to_nearest_school_m', 'school_count_1km', 'primary_school_quality_1km_weighted'
    ],
    'files': {
        'feature_table': str(feature_table_fp),
        'train': str(train_fp),
        'test': str(test_fp),
    },
}
meta_fp = OUTPUT_DIR / f'feature_metadata_{RUN_DATE}.json'
with open(meta_fp, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Saved feature table:', feature_table_fp.name)
print('Saved train:', train_fp.name, '| test:', test_fp.name)
print('Metadata:', meta_fp.name)
print('Rows total/train/test:', len(use_model), len(train_df), len(test_df))

Saved feature table: hdb_feature_table_20260317.csv
Saved train: hdb_feature_train_20260317.csv | test: hdb_feature_test_20260317.csv
Metadata: feature_metadata_20260317.json
Rows total/train/test: 526848 421479 105369


In [20]:

# Validation: check mall counts for Serangoon/Lorong Lew Lian after fix
rows = feat[feat['address_key'].str.contains('LORONG LEW LIAN', na=False)]
print(f"Lorong Lew Lian rows: {len(rows)}")
if len(rows):
    print(rows[['address_key','mall_count_3km','dist_to_nearest_mall_m','mall_weighted_access_3km']].head())
print(f"\nOverall mall_count_3km: mean={feat['mall_count_3km'].mean():.1f}, median={feat['mall_count_3km'].median():.0f}")
print(f"Rows with mall_count_3km == 0: {(feat['mall_count_3km']==0).sum()} / {len(feat)}")


Lorong Lew Lian rows: 0

Overall mall_count_3km: mean=9.5, median=7
Rows with mall_count_3km == 0: 112 / 526848


## Download outputs from Hugging Face

Optional: Sync feature outputs from remote repository.

In [ ]:
from huggingface_hub import snapshot_download, HfApi
from pathlib import Path
import shutil
import os

# USER CONTROL: Set to True to enable download from Hugging Face
DOWNLOAD_OUTPUTS_ENABLED = False

if DOWNLOAD_OUTPUTS_ENABLED:
    # 从环境变量读取 token（安全，不hardcode）
    HF_TOKEN = os.getenv('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('❌ HF_TOKEN not found in environment. Please set it in .env file')
    
    REPO_ID = 'PropertyLens/Resealeflats'
    
    # Define output directory path
    ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
    FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
    
    try:
        print(f'Downloading from {REPO_ID}...')
        repo_dir = snapshot_download(repo_id=REPO_ID, repo_type='dataset', token=HF_TOKEN, cache_dir='/tmp/hf_cache')
        src_outputs = Path(repo_dir) / '02_feature_layer' / 'training' / 'outputs'
        
        if src_outputs.exists():
            # Copy outputs files
            items = list(src_outputs.iterdir())
            for item in items:
                if item.is_file():
                    dest = FEATURE_DIR / item.name
                    shutil.copy2(item, dest)
                    print(f'  ✓ {item.name}')
            print(f'✓ Downloaded {len(items)} items to outputs/')
        else:
            print(f'⚠ No 02_feature_layer/training/outputs found in repository')
    except Exception as e:
        print(f'✗ Download failed: {e}')
else:
    print('Download skipped (DOWNLOAD_OUTPUTS_ENABLED = False)')

## Upload outputs to Hugging Face

Optional: Sync feature outputs back to remote repository after changes.

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone
from pathlib import Path
import os

# USER CONTROL: Set to True to enable upload to Hugging Face
UPLOAD_OUTPUTS_ENABLED = False

if UPLOAD_OUTPUTS_ENABLED:
    # 从环境变量读取 token（安全，不hardcode）
    HF_TOKEN = os.getenv('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('❌ HF_TOKEN not found in environment. Please set it in .env file')
    
    REPO_ID = 'PropertyLens/Resealeflats'
    
    # Define output directory path
    ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
    FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
    OUTPUT_DIR = FEATURE_DIR / 'outputs'
    
    api = HfApi(token=HF_TOKEN)
    timestamp = datetime.now(timezone.utc).isoformat()
    
    try:
        print(f'Uploading outputs to {REPO_ID}...')
        
        # Upload all files in outputs directory
        items = list(OUTPUT_DIR.iterdir())
        success_count = 0
        
        for item in items:
            try:
                if item.is_file():
                    path_in_repo = f'02_feature_layer/training/outputs/{item.name}'
                    api.upload_file(
                        path_or_fileobj=str(item),
                        repo_id=REPO_ID,
                        path_in_repo=path_in_repo,
                        repo_type='dataset',
                        commit_message=f'Update {item.name} at {timestamp}'
                    )
                    print(f'  ✓ {item.name}')
                    success_count += 1
            except Exception as e:
                print(f'  ✗ {item.name}: {e}')
        
        print(f'\n✓ Upload complete: {success_count}/{len(items)} files')
        print(f'Repository: https://huggingface.co/datasets/{REPO_ID}')
        
    except Exception as e:
        print(f'✗ Upload failed: {e}')
else:
    print('Upload skipped (UPLOAD_OUTPUTS_ENABLED = False)')

Uploading outputs to PropertyLens/Resealeflats...


Processing Files (1 / 1): 100%|██████████| 53.2MB / 53.2MB, 5.32MB/s  
New Data Upload: 100%|██████████| 53.2MB / 53.2MB, 5.32MB/s  


  ✓ hdb_feature_test_20260317.csv


Processing Files (1 / 1): 100%|██████████|  266MB /  266MB,  152kB/s  
New Data Upload: 100%|██████████|  262MB /  262MB,  152kB/s  


  ✓ hdb_feature_table_20260317.csv
  ✓ feature_metadata_20260317.json


Processing Files (1 / 1): 100%|██████████|  213MB /  213MB, 8.01MB/s  
New Data Upload: 100%|██████████|  213MB /  213MB, 8.01MB/s  


  ✓ hdb_feature_train_20260317.csv

✓ Upload complete: 4/4 files
Repository: https://huggingface.co/datasets/PropertyLens/Resealeflats
